# headswap — expression chain

**T4 head swap → LivePortrait expression transfer.**

Cell 1 (setup) → Cell 2 (upload) → **Cell 3 (LOAD, once)** → **Cell 4 (RUN, ~21s every time)**.

Cell 3 loads both models into this notebook's kernel and runs one throwaway pass to pay the CUDA warm-up. After that, Cell 4 costs the warm number on **every** run — re-run it as often as you like, or re-run Cell 2 for a new pair and then Cell 4 again.

Only re-run Cell 3 if the runtime restarts.

*(An earlier version ran everything in a subprocess, which freed the models on exit and paid cold start every single time — that is why the same cell produced 100s then 21s.)*

Settings fixed to the arm that worked: LivePortrait runs **after** the swap (before it, T4 regenerates the head and the expression is lost), `driving_multiplier=0.8`, `animation_region=lip`, `face_refine` skipped on bust shots.


In [ ]:
#@title Cell 1 - Setup (run once per runtime)
from pathlib import Path
import subprocess, shutil, os, signal, sys

assert Path("/content").exists(), "Open this notebook in Google Colab."
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"
if not REPO.exists():
    subprocess.run(["git", "clone",
                    "https://github.com/malihashar/headswap_V2.git", str(REPO)],
                   check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH,
                f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"],
               check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

FRESH = not Path("/content/ComfyUI/server.py").exists()
if FRESH:
    print("-> Fresh runtime: installing ComfyUI + Krea2 weights (~20GB, slow)")
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1200:])
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("setup_colab.sh failed")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=True)
    print("\n✓ Restarting kernel (expected). Then run Cell 2.")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("ComfyUI already present - skipping the slow install.")
    print("✓ Ready. Run Cell 2.  (LivePortrait auto-installs in Cell 3.)")


In [ ]:
#@title Cell 2 - Upload a pair
# BODY  = the photo you keep (pose, clothing, background) AND whose
#         expression you want on the final face.
# FACE  = the donor whose identity is transferred in.
import os, uuid
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def grab(role, what):
    print(f"\n=== Upload the {role.upper()} image ({what}) ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} uploaded - re-run this cell.")
    name = next(iter(up))
    Image.open(name).convert("RGB").save(PAIR / f"{role}.png")
    os.remove(name)
    im = Image.open(PAIR / f"{role}.png")
    print(f"  saved {role}: {im.size[0]}x{im.size[1]}")
    return im

body = grab("body", "TARGET: pose/clothes/background + the expression you want")
face = grab("face", "DONOR: the identity to transfer in")
display(body); display(face)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - LOAD models into this kernel (run once per runtime)
import subprocess, sys, os
from pathlib import Path

REPO = Path("/content/headswap_V2")
subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))
os.chdir(REPO)

# LivePortrait (auto-install if the runtime was recycled)
LP = Path("/content/LivePortrait")
if not (LP / "inference.py").exists():
    print("-> installing LivePortrait ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KwaiVGI/LivePortrait", str(LP)], check=True)
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "tyro", "imageio",
                    "imageio-ffmpeg", "rich", "pykalman", "ffmpeg-python"],
                   check=False)
if not (LP / "pretrained_weights" / "liveportrait").exists():
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="KwaiVGI/LivePortrait",
                      local_dir=str(LP / "pretrained_weights"),
                      ignore_patterns=["*animal*"])

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.chain import warmup

PAIR = REPO / "data" / "custom" / "chain_pair"
if not (PAIR / "body.png").exists():
    raise SystemExit("No pair uploaded - run Cell 2 first.")

print("\nLoading models + one throwaway pass (this is the slow part) ...\n")
info = warmup(PAIR / "body.png", PAIR / "face.png", lp_dir=LP)
print(f"\n✓ Models resident in this kernel. Cell 4 is now warm.")


In [ ]:
#@title Cell 4 - RUN (warm: this is the real per-request time)
import time
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown

from headswap.chain import run_chain

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"
OUT = REPO / "results" / "chain_run"

t0 = time.perf_counter()
r = run_chain(PAIR / "body.png", PAIR / "face.png",
              out_dir=OUT, lp_dir="/content/LivePortrait")
wall = time.perf_counter() - t0

print("\n" + "=" * 52)
print(f"  T4 swap        {r['swap_s']:6.1f}s")
print(f"  LivePortrait   {r['lp_s']:6.1f}s   (applied={r['lp_applied']})")
print(f"  TOTAL          {r['total_s']:6.1f}s   (cell wall {wall:.1f}s)")
print("=" * 52)
if not r["was_warm"]:
    print("  NOTE: models were not warmed - run Cell 3 first for a real number.")

display(Markdown("### T4 swap only"))
display(IPImage(filename=str(r["swap_only"])))
display(Markdown("### Final (after LivePortrait)"))
display(IPImage(filename=str(r["final"])))
